# Laboratorio 03.2 — Validación y reintentos acotados

Probaremos respuestas mal formadas e incompatibles con el esquema y diseñaremos una política de reintentos limitada. Las respuestas de este laboratorio son simuladas para evitar llamadas a modelos.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Distinguir errores de sintaxis JSON de errores del contrato Pydantic.
- Convertir errores de validación en diagnósticos útiles para una corrección.
- Implementar un máximo explícito de intentos y una salida de fallo.
- Medir cumplimiento de esquema y tasa de fallos según el presupuesto de reintentos.

## 2. Conceptos

Una respuesta estructurada puede fallar al parsearse o al validar tipos, enums y restricciones. Reintentar puede corregir errores transitorios, pero cada intento consume latencia y, si se usa un LLM, puede costar dinero. Una política debe limitar intentos, evitar bucles y conservar el fallo final para observabilidad; nunca debe aceptar silenciosamente una salida inválida.

## 3. Arquitectura

```mermaid
flowchart TD
    A[Respuesta del modelo] --> B[Validacion Pydantic]
    B -->|valida| C[Objeto tipado]
    B -->|error y quedan intentos| D[Diagnostico acotado]
    D --> E[Nueva respuesta]
    E --> B
    B -->|limite agotado| F[Fallo explicito]
```

## 4. Preparación del entorno

Necesitamos JSON, Pydantic y pandas. No conectamos el bucle a un modelo: una lista controlada de respuestas permite reproducir cada rama y probar el límite de intentos sin costo.

In [ ]:
from enum import Enum

import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

from llm_engineering.environment import preparar_entorno

raiz_curso = preparar_entorno()

## 5. Definición del problema

Un clasificador debe producir una categoría, una confianza acotada y evidencia breve. Las salidas simuladas incluyen JSON truncado, un valor fuera de rango y finalmente un objeto válido. El objetivo es observar el control de flujo, no estimar la capacidad de un modelo.

## 6. Línea base

Sin validación, la aplicación podría consumir cualquier respuesta como texto. Mostramos las tres respuestas candidatas antes de establecer qué acepta el contrato.

In [ ]:
respuestas_simuladas = [
    '{"categoria": "envios", "confianza": 0.9,',
    '{"categoria": "devoluciones", "confianza": 1.4, "evidencia": "Producto abierto"}',
    '{"categoria": "envios", "confianza": 0.92, "evidencia": "El texto indica que el paquete no llego."}',
]
for numero, respuesta in enumerate(respuestas_simuladas, start=1):
    print(f'Candidata {numero}: {respuesta}')

## 7. Implementación

El esquema rechaza propiedades extra, limita confianza y exige evidencia no vacía. La función consume como máximo el número indicado de candidatos, registra solo tipo y campos del error, y devuelve `None` si se agota el presupuesto.

In [ ]:
class Categoria(str, Enum):
    ENVIOS = 'envios'
    DEVOLUCIONES = 'devoluciones'
    PAGOS = 'pagos'


class SalidaClasificacion(BaseModel):
    model_config = ConfigDict(extra='forbid')

    categoria: Categoria
    confianza: float = Field(ge=0.0, le=1.0)
    evidencia: str = Field(min_length=8, max_length=160)

    @field_validator('evidencia')
    @classmethod
    def normalizar_evidencia(cls, valor: str) -> str:
        """Recorta espacios externos de la evidencia."""
        return valor.strip()


def validar_con_limite(max_intentos: int) -> tuple[SalidaClasificacion | None, list[dict[str, object]]]:
    """Valida respuestas simuladas hasta alcanzar un limite de intentos."""
    if max_intentos < 1:
        raise ValueError('max_intentos debe ser al menos uno.')
    traza = []
    for numero, contenido in enumerate(respuestas_simuladas[:max_intentos], start=1):
        try:
            salida = SalidaClasificacion.model_validate_json(contenido)
            traza.append({'intento': numero, 'valido': True, 'tipo_error': None, 'campos': []})
            return salida, traza
        except ValidationError as error:
            errores = error.errors()
            traza.append({
                'intento': numero,
                'valido': False,
                'tipo_error': errores[0]['type'],
                'campos': [str(item) for item in errores[0]['loc']],
            })
    return None, traza

salida_final, traza = validar_con_limite(max_intentos=3)
print('Salida final:', salida_final)
display(pd.DataFrame(traza))

## 8. Experimentos

Variamos únicamente el límite de intentos y reutilizamos la misma secuencia simulada. Esto permite medir el intercambio entre recuperación y trabajo máximo antes de conectar reintentos a llamadas reales.

In [ ]:
filas_experimento = []
for limite in (1, 2, 3):
    salida, intentos = validar_con_limite(limite)
    filas_experimento.append({
        'max_intentos': limite,
        'intentos_consumidos': len(intentos),
        'schema_valid': salida is not None,
        'error_final': intentos[-1]['tipo_error'],
        'llamadas_reales': 0,
    })
tabla_reintentos = pd.DataFrame(filas_experimento)
display(tabla_reintentos)
tabla_reintentos.to_csv(raiz_curso / 'outputs' / '03_02_retry_experiments.csv', index=False)

## 9. Evaluación

`schema_valid` mide si la secuencia produjo un objeto aceptable dentro del límite. La tasa de fallo crece cuando el presupuesto es demasiado bajo para esta secuencia; el máximo de intentos sigue siendo una cota explícita de costo potencial, no una garantía de éxito.

In [ ]:
metricas_reintentos = pd.DataFrame([{
    'experimentos': len(tabla_reintentos),
    'cumplimiento_medio': float(tabla_reintentos['schema_valid'].mean()),
    'tasa_fallo': float((~tabla_reintentos['schema_valid']).mean()),
    'maximo_intentos_configurado': int(tabla_reintentos['max_intentos'].max()),
}])
display(metricas_reintentos)
metricas_reintentos.to_csv(raiz_curso / 'outputs' / '03_02_retry_metrics.csv', index=False)

## 10. Discusión

Los errores pueden proporcionar señales para corregir la siguiente generación, pero enviar detalles sin filtrar puede filtrar datos o instrucciones internas. En producción conviene resumir problemas accionables, imponer límites de tamaño y tiempo, registrar intentos y escalar a un fallo explícito cuando el presupuesto termina.

## 11. Ejercicios

1. Agrega un candidato con una categoría desconocida y registra el tipo de error.
2. Compara límites de cero, uno y dos intentos, definiendo el comportamiento para cero.
3. Añade una pausa exponencial simulada y explica por qué no siempre conviene reintentar.
4. Diseña un mensaje de corrección que no incluya el payload completo.

## 12. Desafío

Conecta este validador a un modelo local usando un máximo de dos regeneraciones. Conserva una línea base sin reintentos, cuenta solicitudes reales y compara tasa de éxito, latencia y costo estimado; deja el flujo remoto desactivado por defecto.

## 13. Ideas clave

- Parsear JSON y validar un esquema son etapas distintas.
- Los errores tipados guían correcciones, pero no deben propagarse sin control.
- Cada política de recuperación necesita un límite explícito.
- El fallo final debe permanecer visible y medible.